In [1]:
import pandas as pd
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler
from utils.preprocess import create_features
from utils.data_split import split_data
from utils.wandb_config import start_run, log_metrics, finish_run
from utils.config import SEED
from utils.metrics import evaluate
import numpy as np
import wandb


In [2]:
df = pd.read_csv('../../../data/creditcard.csv')
df = create_features(df)
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V25,V26,V27,V28,Amount,Class,_log_amount,Hour_from_start_mod24,is_night_proxy,is_business_hours_proxy
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.128539,-0.189115,0.133558,-0.021053,149.62,0,5.014760,0,1,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0.167170,0.125895,-0.008983,0.014724,2.69,0,1.305626,0,1,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0,5.939276,0,1,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0.647376,-0.221929,0.062723,0.061458,123.50,0,4.824306,0,1,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.206010,0.502292,0.219422,0.215153,69.99,0,4.262539,0,1,0


In [3]:
features = [c for c in df.columns if c.startswith("V")] + [
    'Amount','_log_amount',
    'Hour_from_start_mod24','is_night_proxy','is_business_hours_proxy'
]
target = "Class"

In [4]:
X_train, y_train, X_val, y_val, X_test, y_test = split_data(df, features, target)

X_train: (181584, 33) y_train: (181584,)
X_val: (45396, 33) y_val: (45396,)
X_test: (56746, 33) y_test: (56746,)
Fraud rate in train: 0.001910961318177813
Fraud rate in test: 0.0013040566735981391


In [5]:
run = start_run(
    model_name="lightgbm",
    config={
        "n_estimators":500,
        "learning_rate":0.05,
        "num_leaves":31 
    }
)


Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: 2351050051huy (2351050051huy-tr-ng-i-h-c-m-th-nh-ph-h-ch-minh). Use `wandb login --relogin` to force relogin
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


In [6]:
pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
lgbm_pipe = ImbPipeline(steps=[
    ("model", LGBMClassifier(
        n_estimators=wandb.config.n_estimators,
        learning_rate=wandb.config.learning_rate,
        num_leaves=wandb.config.num_leaves,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        scale_pos_weight=neg/max(pos, 1),
    ))
])

lgbm_pipe.fit(X_train, y_train)

val_lgb_proba  = lgbm_pipe.predict_proba(X_val)[:,1]
test_lgb_proba = lgbm_pipe.predict_proba(X_test)[:,1]

[LightGBM] [Info] Number of positive: 347, number of negative: 181237
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014050 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7678
[LightGBM] [Info] Number of data points in the train set: 181584, number of used features: 33
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001911 -> initscore=-6.258236
[LightGBM] [Info] Start training from score -6.258236
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

In [7]:
metrics = evaluate(y_test, test_lgb_proba, thr=0.5)

In [8]:
print(metrics)

{'threshold': 0.5, 'precision': 0.02311912225705329, 'recall': 0.7972972972972973, 'f1': 0.04493526275704494, 'roc_auc': 0.876632080096754, 'auprc': 0.018704375224423028, 'brier': 0.04419694778028475, 'tp': 59, 'fp': 2493, 'fn': 15, 'tn': 54179}


In [9]:
wandb.log(metrics)
wandb.finish()

wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
